# Phase 7 — Results Visualisation and Interpretation

This notebook converts the Phase 6 PPO-versus-random evaluation results into
clear figures, summary tables and dissertation-ready interpretation.

## Required upload

Upload:

`Farmer_RL_Phase6_Results.zip`

## Outputs

- Report-quality PNG graphs
- Combined model-performance tables
- Crop-selection analysis
- Savings and reward interpretation
- A ready-to-use results narrative
- `Farmer_RL_Phase7_Results.zip`

In [ ]:
# Install the libraries required by this notebook.
!pip -q install pandas matplotlib

In [ ]:
import json
import shutil
import warnings
import zipfile
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=DeprecationWarning)

PHASE6_FOLDER = Path("phase6_files")
TABLES_FOLDER = Path("phase7_outputs/tables")
GRAPHS_FOLDER = Path("phase7_outputs/graphs")
TEXT_FOLDER = Path("phase7_outputs/interpretation")

for folder in [
    PHASE6_FOLDER,
    TABLES_FOLDER,
    GRAPHS_FOLDER,
    TEXT_FOLDER,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Phase 7 folders created.")

## 1. Upload and extract the Phase 6 results

In [ ]:
from google.colab import files

print("Upload Farmer_RL_Phase6_Results.zip")
uploaded = files.upload()

zip_files = [
    filename
    for filename in uploaded
    if filename.lower().endswith(".zip")
]

if not zip_files:
    raise FileNotFoundError(
        "No ZIP file was uploaded. "
        "Upload Farmer_RL_Phase6_Results.zip."
    )

phase6_zip = zip_files[0]

if PHASE6_FOLDER.exists():
    shutil.rmtree(PHASE6_FOLDER)

PHASE6_FOLDER.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(phase6_zip, "r") as archive:
    archive.extractall(PHASE6_FOLDER)

print("Extracted:", phase6_zip)

In [ ]:
def find_file(filename: str):
    matches = list(PHASE6_FOLDER.rglob(filename))
    return matches[0] if matches else None


required_files = {
    "ppo_results": find_file(
        "ppo_evaluation_results.csv"
    ),
    "random_results": find_file(
        "random_evaluation_results.csv"
    ),
    "ppo_yearly": find_file(
        "ppo_yearly_decisions.csv"
    ),
    "random_yearly": find_file(
        "random_yearly_decisions.csv"
    ),
    "comparison": find_file(
        "ppo_vs_random_comparison.csv"
    ),
}

missing_files = [
    name
    for name, path in required_files.items()
    if path is None
]

if missing_files:
    raise FileNotFoundError(
        "The Phase 6 ZIP is missing required files: "
        + ", ".join(missing_files)
    )

for name, path in required_files.items():
    print(f"{name}: {path}")

## 2. Load and validate the evaluation data

In [ ]:
ppo_results = pd.read_csv(
    required_files["ppo_results"]
)
random_results = pd.read_csv(
    required_files["random_results"]
)
ppo_yearly = pd.read_csv(
    required_files["ppo_yearly"]
)
random_yearly = pd.read_csv(
    required_files["random_yearly"]
)
phase6_comparison = pd.read_csv(
    required_files["comparison"]
)

required_episode_columns = {
    "total_reward",
    "final_savings",
    "total_net_income",
    "average_annual_income",
    "years_survived",
    "bankrupt",
}

required_yearly_columns = {
    "year",
    "crop",
    "net_income",
    "savings",
    "reward",
}

for name, frame in [
    ("PPO results", ppo_results),
    ("Random results", random_results),
]:
    missing = required_episode_columns.difference(
        frame.columns
    )
    if missing:
        raise ValueError(
            f"{name} is missing columns: {sorted(missing)}"
        )

for name, frame in [
    ("PPO yearly data", ppo_yearly),
    ("Random yearly data", random_yearly),
]:
    missing = required_yearly_columns.difference(
        frame.columns
    )
    if missing:
        raise ValueError(
            f"{name} is missing columns: {sorted(missing)}"
        )

ppo_results["policy"] = "PPO"
random_results["policy"] = "Random"
ppo_yearly["policy"] = "PPO"
random_yearly["policy"] = "Random"

episode_results = pd.concat(
    [ppo_results, random_results],
    ignore_index=True,
)

yearly_results = pd.concat(
    [ppo_yearly, random_yearly],
    ignore_index=True,
)

for frame in [
    episode_results,
    yearly_results,
]:
    if "bankrupt" in frame.columns:
        frame["bankrupt"] = (
            frame["bankrupt"]
            .astype(str)
            .str.lower()
            .map({"true": True, "false": False})
            .fillna(frame["bankrupt"])
        )

print("PPO episodes:", len(ppo_results))
print("Random episodes:", len(random_results))
print("PPO yearly decisions:", len(ppo_yearly))
print("Random yearly decisions:", len(random_yearly))

display(episode_results.head())

## 3. Create the main statistical summary

In [ ]:
summary_metrics = [
    "total_reward",
    "final_savings",
    "total_net_income",
    "average_annual_income",
    "years_survived",
]

summary_rows = []

for policy, group in episode_results.groupby("policy"):
    row = {
        "Policy": policy,
        "Episodes": len(group),
        "Bankruptcy Rate (%)": (
            pd.to_numeric(
                group["bankrupt"],
                errors="coerce",
            ).mean()
            * 100.0
        ),
    }

    for metric in summary_metrics:
        values = pd.to_numeric(
            group[metric],
            errors="coerce",
        )
        display_name = metric.replace("_", " ").title()
        row[f"Mean {display_name}"] = values.mean()
        row[f"Std {display_name}"] = values.std(ddof=1)
        row[f"Median {display_name}"] = values.median()

    summary_rows.append(row)

summary_table = pd.DataFrame(summary_rows)

summary_table.to_csv(
    TABLES_FOLDER / "main_statistical_summary.csv",
    index=False,
)

display(summary_table.round(3))

In [ ]:
ppo_summary = summary_table[
    summary_table["Policy"] == "PPO"
].iloc[0]

random_summary = summary_table[
    summary_table["Policy"] == "Random"
].iloc[0]

comparison_rows = []

for metric in summary_metrics:
    title = metric.replace("_", " ").title()

    ppo_value = float(
        ppo_summary[f"Mean {title}"]
    )
    random_value = float(
        random_summary[f"Mean {title}"]
    )

    difference = ppo_value - random_value

    percentage = (
        np.nan
        if np.isclose(random_value, 0.0)
        else difference / abs(random_value) * 100.0
    )

    comparison_rows.append({
        "Metric": title,
        "PPO Mean": ppo_value,
        "Random Mean": random_value,
        "Absolute Difference": difference,
        "PPO Difference (%)": percentage,
        "Better Policy": (
            "PPO" if ppo_value > random_value else "Random"
        ),
    })

comparison_table = pd.DataFrame(comparison_rows)

comparison_table.to_csv(
    TABLES_FOLDER / "model_comparison_summary.csv",
    index=False,
)

display(comparison_table.round(3))

## 4. Average model-performance comparison

In [ ]:
mean_values = (
    episode_results.groupby("policy")[
        [
            "total_reward",
            "final_savings",
            "total_net_income",
            "years_survived",
        ]
    ]
    .mean()
)

metric_labels = {
    "total_reward": "Total reward",
    "final_savings": "Final savings",
    "total_net_income": "Total net income",
    "years_survived": "Years survived",
}

for metric, label in metric_labels.items():
    plt.figure(figsize=(8, 5))
    values = [
        mean_values.loc["PPO", metric],
        mean_values.loc["Random", metric],
    ]

    bars = plt.bar(
        ["PPO", "Random"],
        values,
    )

    plt.title(f"Average {label}: PPO vs Random")
    plt.xlabel("Decision policy")
    plt.ylabel(label)

    for bar, value in zip(bars, values):
        plt.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height(),
            f"{value:,.2f}",
            ha="center",
            va="bottom",
        )

    plt.tight_layout()
    plt.savefig(
        GRAPHS_FOLDER / f"average_{metric}.png",
        dpi=300,
        bbox_inches="tight",
    )
    plt.show()

## 5. Episode reward and savings distributions

In [ ]:
plt.figure(figsize=(9, 6))

plt.boxplot(
    [
        ppo_results["total_reward"],
        random_results["total_reward"],
    ],
    tick_labels=["PPO", "Random"],
    showmeans=True,
)

plt.title("Distribution of Total Episode Rewards")
plt.xlabel("Decision policy")
plt.ylabel("Total reward")
plt.tight_layout()

plt.savefig(
    GRAPHS_FOLDER / "reward_distribution_boxplot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
plt.figure(figsize=(9, 6))

plt.boxplot(
    [
        ppo_results["final_savings"],
        random_results["final_savings"],
    ],
    tick_labels=["PPO", "Random"],
    showmeans=True,
)

plt.title("Distribution of Final Farmer Savings")
plt.xlabel("Decision policy")
plt.ylabel("Final savings")
plt.tight_layout()

plt.savefig(
    GRAPHS_FOLDER / "savings_distribution_boxplot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 6. Savings and income trends across planning years

In [ ]:
average_yearly = (
    yearly_results.groupby(["policy", "year"])
    .agg(
        average_savings=("savings", "mean"),
        average_net_income=("net_income", "mean"),
        average_reward=("reward", "mean"),
    )
    .reset_index()
)

average_yearly.to_csv(
    TABLES_FOLDER / "average_yearly_performance.csv",
    index=False,
)

display(average_yearly.head())

In [ ]:
plt.figure(figsize=(10, 6))

for policy, group in average_yearly.groupby("policy"):
    plt.plot(
        group["year"],
        group["average_savings"],
        marker="o",
        label=policy,
    )

plt.title("Average Savings Across the Planning Horizon")
plt.xlabel("Planning year")
plt.ylabel("Average savings")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    GRAPHS_FOLDER / "average_savings_by_year.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

for policy, group in average_yearly.groupby("policy"):
    plt.plot(
        group["year"],
        group["average_net_income"],
        marker="o",
        label=policy,
    )

plt.title("Average Net Income Across the Planning Horizon")
plt.xlabel("Planning year")
plt.ylabel("Average net income")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    GRAPHS_FOLDER / "average_net_income_by_year.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
plt.figure(figsize=(10, 6))

for policy, group in average_yearly.groupby("policy"):
    plt.plot(
        group["year"],
        group["average_reward"],
        marker="o",
        label=policy,
    )

plt.title("Average Reward Across the Planning Horizon")
plt.xlabel("Planning year")
plt.ylabel("Average reward")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

plt.savefig(
    GRAPHS_FOLDER / "average_reward_by_year.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 7. Crop-selection frequency and crop performance

In [ ]:
crop_frequency = (
    yearly_results.groupby(["policy", "crop"])
    .size()
    .rename("selection_count")
    .reset_index()
)

crop_frequency["selection_percentage"] = (
    crop_frequency.groupby("policy")[
        "selection_count"
    ]
    .transform(lambda values: values / values.sum() * 100.0)
)

crop_frequency.to_csv(
    TABLES_FOLDER / "crop_selection_frequency.csv",
    index=False,
)

crop_frequency_pivot = crop_frequency.pivot(
    index="crop",
    columns="policy",
    values="selection_percentage",
).fillna(0.0)

display(
    crop_frequency_pivot.sort_values(
        "PPO",
        ascending=False,
    ).round(2)
)

In [ ]:
crop_frequency_pivot.plot(
    kind="bar",
    figsize=(13, 6),
)

plt.title("Crop Selection Frequency: PPO vs Random")
plt.xlabel("Crop")
plt.ylabel("Selection frequency (%)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Policy")
plt.tight_layout()

plt.savefig(
    GRAPHS_FOLDER / "crop_selection_frequency.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
crop_performance = (
    yearly_results.groupby(["policy", "crop"])
    .agg(
        selections=("crop", "size"),
        average_net_income=("net_income", "mean"),
        average_reward=("reward", "mean"),
        average_savings=("savings", "mean"),
    )
    .reset_index()
)

crop_performance.to_csv(
    TABLES_FOLDER / "crop_performance_analysis.csv",
    index=False,
)

ppo_crop_performance = (
    crop_performance[
        crop_performance["policy"] == "PPO"
    ]
    .sort_values(
        "average_net_income",
        ascending=False,
    )
)

display(ppo_crop_performance.round(2))

In [ ]:
top_crops = ppo_crop_performance.head(10)

plt.figure(figsize=(10, 6))
plt.barh(
    top_crops["crop"][::-1],
    top_crops["average_net_income"][::-1],
)

plt.title("Top PPO-Selected Crops by Average Net Income")
plt.xlabel("Average net income")
plt.ylabel("Crop")
plt.tight_layout()

plt.savefig(
    GRAPHS_FOLDER / "top_ppo_crops_by_income.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 8. Bankruptcy and farming-risk analysis

In [ ]:
bankruptcy_summary = (
    episode_results.groupby("policy")["bankrupt"]
    .agg(["count", "sum", "mean"])
    .reset_index()
)

bankruptcy_summary = bankruptcy_summary.rename(
    columns={
        "count": "episodes",
        "sum": "bankrupt_episodes",
        "mean": "bankruptcy_rate",
    }
)

bankruptcy_summary["bankruptcy_rate_percent"] = (
    bankruptcy_summary["bankruptcy_rate"] * 100.0
)

bankruptcy_summary.to_csv(
    TABLES_FOLDER / "bankruptcy_analysis.csv",
    index=False,
)

display(bankruptcy_summary.round(3))

In [ ]:
plt.figure(figsize=(8, 5))

rates = bankruptcy_summary.set_index("policy")[
    "bankruptcy_rate_percent"
]

bars = plt.bar(
    ["PPO", "Random"],
    [rates.get("PPO", 0.0), rates.get("Random", 0.0)],
)

plt.title("Bankruptcy Rate by Decision Policy")
plt.xlabel("Decision policy")
plt.ylabel("Bankruptcy rate (%)")

for bar in bars:
    value = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:.2f}%",
        ha="center",
        va="bottom",
    )

plt.tight_layout()
plt.savefig(
    GRAPHS_FOLDER / "bankruptcy_rate.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 9. Automatically interpret the results

In [ ]:
def percentage_difference(
    ppo_value: float,
    random_value: float,
):
    if np.isclose(random_value, 0.0):
        return np.nan
    return (
        (ppo_value - random_value)
        / abs(random_value)
        * 100.0
    )


ppo_reward = float(
    ppo_results["total_reward"].mean()
)
random_reward = float(
    random_results["total_reward"].mean()
)

ppo_savings = float(
    ppo_results["final_savings"].mean()
)
random_savings = float(
    random_results["final_savings"].mean()
)

ppo_income = float(
    ppo_results["total_net_income"].mean()
)
random_income = float(
    random_results["total_net_income"].mean()
)

ppo_survival = float(
    ppo_results["years_survived"].mean()
)
random_survival = float(
    random_results["years_survived"].mean()
)

ppo_bankruptcy = float(
    pd.to_numeric(
        ppo_results["bankrupt"],
        errors="coerce",
    ).mean()
    * 100.0
)
random_bankruptcy = float(
    pd.to_numeric(
        random_results["bankrupt"],
        errors="coerce",
    ).mean()
    * 100.0
)

reward_change = percentage_difference(
    ppo_reward,
    random_reward,
)
savings_change = percentage_difference(
    ppo_savings,
    random_savings,
)
income_change = percentage_difference(
    ppo_income,
    random_income,
)

reward_winner = (
    "PPO" if ppo_reward > random_reward else "random policy"
)
savings_winner = (
    "PPO" if ppo_savings > random_savings else "random policy"
)

top_ppo_crop = (
    crop_frequency[
        crop_frequency["policy"] == "PPO"
    ]
    .sort_values(
        "selection_percentage",
        ascending=False,
    )
    .iloc[0]
)

top_income_crop = (
    ppo_crop_performance.iloc[0]
)

interpretation = f'''
# Phase 7 Results Interpretation

## Overall performance

The Proximal Policy Optimisation agent achieved an average total reward of
{ppo_reward:,.3f}, while the random crop-selection policy achieved
{random_reward:,.3f}. This represents a PPO difference of
{reward_change:,.2f}% relative to the random baseline. Based on average
episode reward, the better-performing strategy was the {reward_winner}.

## Farmer savings

The PPO policy produced average final savings of {ppo_savings:,.2f}, compared
with {random_savings:,.2f} under random crop selection. The difference was
{savings_change:,.2f}%. The policy with the higher average final savings was
the {savings_winner}.

## Net income

Average total net income was {ppo_income:,.2f} for PPO and
{random_income:,.2f} for the random policy. The PPO difference relative to
the baseline was {income_change:,.2f}%.

## Farm survival and financial risk

The PPO agent survived for an average of {ppo_survival:,.2f} years, while the
random policy survived for {random_survival:,.2f} years. The PPO bankruptcy
rate was {ppo_bankruptcy:,.2f}%, compared with {random_bankruptcy:,.2f}% for
random crop selection.

## Crop-selection behaviour

The crop most frequently selected by PPO was
{top_ppo_crop["crop"]}, accounting for
{top_ppo_crop["selection_percentage"]:,.2f}% of PPO decisions. The PPO crop
with the highest observed average net income was
{top_income_crop["crop"]}, with an average net income of
{top_income_crop["average_net_income"]:,.2f}.

## Interpretation

The comparison indicates whether PPO learned a decision strategy that is more
effective than choosing crops randomly. Higher rewards, savings and net income
show improved economic performance. A lower bankruptcy rate indicates better
financial risk management. Crop-frequency results show which crops the agent
preferred under the simulated rainfall, market-price and crop-rotation
conditions.

These results should be interpreted within the limitations of the dataset and
the custom FarmEnv assumptions. The model estimates decisions from historical
data and simulated uncertainty; it does not guarantee real-world farm profit.
Future work should include region-specific soil information, weather
forecasts, crop demand, water availability and independent field validation.
'''.strip()

interpretation_path = (
    TEXT_FOLDER / "phase7_results_interpretation.md"
)
interpretation_path.write_text(
    interpretation,
    encoding="utf-8",
)

display(Markdown(interpretation))

## 10. Generate a concise dissertation results paragraph

In [ ]:
performance_direction = (
    "outperformed"
    if ppo_reward > random_reward
    else "did not outperform"
)

dissertation_paragraph = f'''
The experimental evaluation compared the trained PPO crop-planning agent with
a random crop-selection baseline across {len(ppo_results)} evaluation
episodes. The PPO agent obtained an average cumulative reward of
{ppo_reward:,.3f}, whereas the random policy achieved {random_reward:,.3f}.
Therefore, PPO {performance_direction} random selection in terms of cumulative
reward. PPO also produced average final savings of {ppo_savings:,.2f} and
average total net income of {ppo_income:,.2f}, compared with
{random_savings:,.2f} and {random_income:,.2f}, respectively, for the random
baseline. The bankruptcy rate was {ppo_bankruptcy:,.2f}% for PPO and
{random_bankruptcy:,.2f}% for random selection. These findings indicate the
extent to which the learned policy improved long-term crop-planning decisions
under the financial, rainfall, price-volatility and crop-rotation conditions
represented in the custom reinforcement-learning environment.
'''.strip()

(
    TEXT_FOLDER / "dissertation_results_paragraph.txt"
).write_text(
    dissertation_paragraph,
    encoding="utf-8",
)

print(dissertation_paragraph)

## 11. Create the Phase 7 summary report

In [ ]:
summary_report = {
    "number_of_ppo_episodes": int(
        len(ppo_results)
    ),
    "number_of_random_episodes": int(
        len(random_results)
    ),
    "ppo_average_reward": ppo_reward,
    "random_average_reward": random_reward,
    "ppo_average_final_savings": ppo_savings,
    "random_average_final_savings": random_savings,
    "ppo_average_total_net_income": ppo_income,
    "random_average_total_net_income": random_income,
    "ppo_average_years_survived": ppo_survival,
    "random_average_years_survived": random_survival,
    "ppo_bankruptcy_rate_percent": ppo_bankruptcy,
    "random_bankruptcy_rate_percent": random_bankruptcy,
    "most_selected_ppo_crop": str(
        top_ppo_crop["crop"]
    ),
    "most_selected_ppo_crop_percent": float(
        top_ppo_crop["selection_percentage"]
    ),
    "highest_income_ppo_crop": str(
        top_income_crop["crop"]
    ),
    "highest_crop_average_net_income": float(
        top_income_crop["average_net_income"]
    ),
}

(
    TABLES_FOLDER / "phase7_summary.json"
).write_text(
    json.dumps(summary_report, indent=2),
    encoding="utf-8",
)

print(
    json.dumps(
        summary_report,
        indent=2,
    )
)

## 12. Download all Phase 7 outputs

In [ ]:
from google.colab import files

phase7_zip = "Farmer_RL_Phase7_Results.zip"

with zipfile.ZipFile(
    phase7_zip,
    "w",
    zipfile.ZIP_DEFLATED,
) as archive:
    for file_path in Path(
        "phase7_outputs"
    ).rglob("*"):
        if file_path.is_file():
            archive.write(
                file_path,
                file_path.as_posix(),
            )

print("Created:", phase7_zip)
files.download(phase7_zip)

## Phase 7 completion checklist

After running the notebook, keep:

- `Farmer_RL_Phase7_Results.zip`
- The PNG graphs for your dissertation
- `main_statistical_summary.csv`
- `model_comparison_summary.csv`
- `phase7_results_interpretation.md`
- `dissertation_results_paragraph.txt`

## Next phase

**Phase 8 — Farmer Crop Recommendation Web Application**

The web application will load the trained PPO model and allow a farmer to
enter conditions and receive a crop recommendation.